Setup

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, least, sum as Fsum
import os

spark = (SparkSession.builder
         .appName("stress-test-solution")
         .config("spark.sql.adaptive.enabled","true")
         .getOrCreate())

Part A — Read

In [4]:
base_dir = os.getcwd()  # current notebook folder
csv_path = os.path.abspath(os.path.join(base_dir, "..", "..", "data", "stress_banking_demo"))

exposures = (spark.read
             .option("header", True).option("inferSchema", True)
             .csv(f"{csv_path}/exposures.csv"))

scenarios = (spark.read
             .option("header", True).option("inferSchema", True)
             .csv(f"{csv_path}/scenarios.csv"))


Task A2. Inspect a few rows:

In [5]:
exposures.show(5, truncate=False)
scenarios.show()

+-------+-------+-------+-------+-----+----+-------------+
|loan_id|segment|country|ead    |pd   |lgd |interest_rate|
+-------+-------+-------+-------+-----+----+-------------+
|L001   |Corp   |FR     |50000  |0.015|0.35|0.025        |
|L002   |Retail |FR     |1000000|0.03 |0.3 |0.031        |
|L003   |SME    |FR     |50000  |0.005|0.35|0.025        |
|L004   |Corp   |NL     |50000  |0.03 |0.35|0.031        |
|L005   |SME    |DE     |250000 |0.03 |0.4 |0.022        |
+-------+-------+-------+-------+-----+----+-------------+
only showing top 5 rows

+-------------+-------------+---------+--------------+
|     scenario|pd_multiplier|lgd_addon|rate_shock_bps|
+-------------+-------------+---------+--------------+
|     Baseline|          1.0|      0.0|             0|
|      Adverse|          1.5|     0.05|           150|
|       Severe|          2.5|      0.1|           300|
|Idiosyncratic|          2.0|     0.02|            50|
+-------------+-------------+---------+--------------+



Part B — Transform

Task B1. Join exposures with every scenario (cross join):

In [6]:
x = exposures.crossJoin(scenarios)


Task B2. Create stressed metrics and expected loss (EL):

In [7]:
from pyspark.sql.functions import col, lit, least

result = (x
  .withColumn("stressed_pd",  (col("pd") * col("pd_multiplier")).cast("double"))
  .withColumn("stressed_lgd", least(col("lgd") + col("lgd_addon"), lit(1.0)))
  .withColumn("stressed_rate", (col("interest_rate") + col("rate_shock_bps")/10000.0).cast("double"))
  .withColumn("expected_loss", col("ead") * col("stressed_pd") * col("stressed_lgd"))
)


Task B3. Preview:

In [8]:
result.select("loan_id","scenario","segment","ead","stressed_pd","stressed_lgd","expected_loss")\
      .show(truncate=False)


+-------+-------------+-------+-------+-----------+-------------------+------------------+
|loan_id|scenario     |segment|ead    |stressed_pd|stressed_lgd       |expected_loss     |
+-------+-------------+-------+-------+-----------+-------------------+------------------+
|L001   |Baseline     |Corp   |50000  |0.015      |0.35               |262.5             |
|L001   |Adverse      |Corp   |50000  |0.0225     |0.39999999999999997|449.99999999999994|
|L001   |Severe       |Corp   |50000  |0.0375     |0.44999999999999996|843.7499999999999 |
|L001   |Idiosyncratic|Corp   |50000  |0.03       |0.37               |555.0             |
|L002   |Baseline     |Retail |1000000|0.03       |0.3                |9000.0            |
|L002   |Adverse      |Retail |1000000|0.045      |0.35               |15749.999999999998|
|L002   |Severe       |Retail |1000000|0.075      |0.4                |30000.0           |
|L002   |Idiosyncratic|Retail |1000000|0.06       |0.32               |19200.0           |

Aggregate

Task C1. Aggregate by (scenario, segment):

In [9]:
from pyspark.sql.functions import sum as Fsum

agg = (result.groupBy("scenario","segment")
       .agg(
           Fsum("ead").alias("total_ead"),
           Fsum("expected_loss").alias("total_expected_loss")
       )
       .withColumn("EL_over_EAD", col("total_expected_loss")/col("total_ead"))
)
agg.orderBy("scenario","segment").show(truncate=False)


+-------------+-------+---------+-------------------+--------------------+
|scenario     |segment|total_ead|total_expected_loss|EL_over_EAD         |
+-------------+-------+---------+-------------------+--------------------+
|Adverse      |Corp   |100000   |1349.9999999999998 |0.013499999999999998|
|Adverse      |Retail |1200000  |18112.5            |0.01509375          |
|Adverse      |SME    |600000   |6562.5             |0.0109375           |
|Baseline     |Corp   |100000   |787.5              |0.007875            |
|Baseline     |Retail |1200000  |10387.5            |0.00865625          |
|Baseline     |SME    |600000   |3875.0             |0.006458333333333333|
|Idiosyncratic|Corp   |100000   |1665.0             |0.01665             |
|Idiosyncratic|Retail |1200000  |22125.0            |0.0184375           |
|Idiosyncratic|SME    |600000   |8150.000000000001  |0.013583333333333334|
|Severe       |Corp   |100000   |2531.2499999999995 |0.025312499999999995|
|Severe       |Retail |12

Part D — Write

Task D1. Write loan-level results to Parquet.
Task D2. Write aggregated results to CSV.

In [ ]:
out_base = os.path.abspath(os.path.join(base_dir, "..", "..", "data", "stress_banking_demo","out"))

(result
 .select("loan_id","scenario","segment","country","ead",
         "stressed_pd","stressed_lgd","stressed_rate","expected_loss")
 .coalesce(2)  # demo only
 .write.mode("overwrite").parquet(f"{out_base}/loan_level_parquet"))

(agg
 .coalesce(2)  # demo only
 .write.mode("overwrite").option("header", True)
 .csv(f"{out_base}/agg_by_segment_scenario_csv"))

print("Wrote:", out_base)


Wrote: c:\Users\mehdi\JupyterNootebok\wytasoft-pyspark-training-lab\data\stress_banking_demo\out


In [12]:
spark.read.parquet(f"{out_base}/loan_level_parquet").show(5)
spark.read.option("header",True).csv(f"{out_base}/agg_by_segment_scenario_csv").show()


+-------+-------------+-------+-------+-------+-----------+-------------------+--------------------+------------------+
|loan_id|     scenario|segment|country|    ead|stressed_pd|       stressed_lgd|       stressed_rate|     expected_loss|
+-------+-------------+-------+-------+-------+-----------+-------------------+--------------------+------------------+
|   L001|     Baseline|   Corp|     FR|  50000|      0.015|               0.35|               0.025|             262.5|
|   L001|      Adverse|   Corp|     FR|  50000|     0.0225|0.39999999999999997|                0.04|449.99999999999994|
|   L001|       Severe|   Corp|     FR|  50000|     0.0375|0.44999999999999996|               0.055| 843.7499999999999|
|   L001|Idiosyncratic|   Corp|     FR|  50000|       0.03|               0.37|0.030000000000000002|             555.0|
|   L002|     Baseline| Retail|     FR|1000000|       0.03|                0.3|               0.031|            9000.0|
+-------+-------------+-------+-------+-